# 01 — Dataset Verification

## What this stage does

Before training a machine-learning model, first establish what measurements actually exist. This notebook checks the Paderborn raw-data folders, verifies the expected bearing IDs and counts the MATLAB recordings in each bearing folder.

## Dataset organisation

The experiment is organised around **physical bearings**, not just an unordered list of signal files. The 32 expected bearing IDs are grouped into four condition categories:

- Healthy: 6 bearing IDs
- Outer-race damage: 12 bearing IDs
- Inner-race damage: 11 bearing IDs
- Combined (Both) damage: 3 bearing IDs

The expected 80 recordings per bearing correspond to four operating-condition identifiers with 20 measurements for each bearing-condition combination:

- `N15_M07_F10`
- `N09_M07_F10`
- `N15_M01_F10`
- `N15_M07_F04`

Therefore, the expected file count is:

`32 bearings × 4 operating conditions × 20 measurements = 2,560 MAT files`

This notebook verifies folder and file counts. It does not yet extract vibration features or train a model.

## What is inside a MAT file?

The original recordings are MATLAB `.mat` files. The Paderborn files contain a nested MATLAB measurement structure with multiple named channels; they are not simply a flat array called `signal`. The separate inference module inspects the channel list, finds the channel named `vibration_1`, and extracts its `Data` array. It also reads the named `speed` channel to obtain measured rotational speed.

The project's inference configuration uses a **nominal 64 kHz sampling-rate assumption**. It preserves that original assumption and does not correct timestamps or resample recordings.

## Processing note

During feature-table construction, one source file, `N15_M01_F10_KA08_2.mat`, caused a MATLAB matrix-reading error. The processing workflow logged the failed file and continued. The V2 project report records 2,559 successfully generated feature rows from the expected 2,560 recordings. That 2,559 figure belongs to the later feature-building stage, not to the folder-count code in this notebook.

Place the original Paderborn folders under `data/raw/`. Raw recordings are intentionally excluded from Git.

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# STAGE 1: DATASET VERIFICATION
# ============================================================

RAW_DIR = Path("../data/raw")

healthy_bearings = [
    "K001", "K002", "K003", "K004", "K005", "K006"
]

outer_bearings = [
    "KA01", "KA03", "KA04", "KA05", "KA06",
    "KA07", "KA08", "KA09", "KA15", "KA16",
    "KA22", "KA30"
]

inner_bearings = [
    "KI01", "KI03", "KI04", "KI05", "KI07",
    "KI08", "KI14", "KI16", "KI17", "KI18", "KI21"
]

both_bearings = [
    "KB23", "KB24", "KB27"
]

expected_bearings = (
    healthy_bearings
    + outer_bearings
    + inner_bearings
    + both_bearings
)

print("Expected bearings:", len(expected_bearings))

existing_bearings = sorted([
    p.name
    for p in RAW_DIR.iterdir()
    if p.is_dir()
])

print("Found bearings:", len(existing_bearings))

missing = sorted(
    set(expected_bearings) - set(existing_bearings)
)

unexpected = sorted(
    set(existing_bearings) - set(expected_bearings)
)

print("\nMissing folders:")
print(missing if missing else "None")

print("\nUnexpected folders:")
print(unexpected if unexpected else "None")

In [ ]:
# ============================================================
# COUNT .MAT FILES
# ============================================================

records = []

for bearing in expected_bearings:

    bearing_dir = RAW_DIR / bearing

    if bearing in healthy_bearings:
        category = "healthy"
    elif bearing in outer_bearings:
        category = "outer"
    elif bearing in inner_bearings:
        category = "inner"
    else:
        category = "both"

    if not bearing_dir.exists():
        records.append({
            "bearing": bearing,
            "category": category,
            "mat_files": 0,
            "status": "MISSING FOLDER"
        })
        continue

    mat_files = list(bearing_dir.glob("*.mat"))

    records.append({
        "bearing": bearing,
        "category": category,
        "mat_files": len(mat_files),
        "status": "OK" if len(mat_files) == 80 else "CHECK"
    })

verification_df = pd.DataFrame(records)

display(verification_df)

In [ ]:
# ============================================================
# VERIFICATION SUMMARY
# ============================================================

print("Total bearings:", len(verification_df))
print("Total .mat files:", verification_df["mat_files"].sum())

print("\nBearings by category:")
print(
    verification_df["category"]
    .value_counts()
)

print("\nFile count distribution:")
print(
    verification_df["mat_files"]
    .value_counts()
    .sort_index()
)

print("\nBearings requiring attention:")
display(
    verification_df[
        verification_df["status"] != "OK"
    ]
)